# 03 RL Environment and PPO Training (Long/Flat)

This notebook trains a PPO agent over weekend decision dates using XGBoost-augmented states.

- Action space: 4 discrete actions (both flat, AMZN long, MSFT long, both long)
- Constraints: long/flat only; max gross exposure 1.0 (0.5 each when both long)
- Reward: net return minus transaction cost and optional drawdown penalty


In [ ]:
from pathlib import Path
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import gymnasium as gym
from gymnasium import spaces

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

SEED = 42
np.random.seed(SEED)

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
OUT_DIR = PROJECT_ROOT / "research_outputs" / "weekend_rl_xgb"
TABLE_DIR = OUT_DIR / "tables"
MODEL_DIR = OUT_DIR / "models"
for p in [TABLE_DIR, MODEL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

xgb_panel = pd.read_parquet(OUT_DIR / "xgb_predictions_panel.parquet").sort_values(["date_decision", "ticker"])
xgb_panel.head()


In [ ]:
# Build date-level panel with both tickers in one row.
def to_date_level(panel: pd.DataFrame) -> pd.DataFrame:
    panel = panel.copy()
    panel["ticker"] = panel["ticker"].str.upper()

    base_cols = ["date_decision", "ticker", "y_true_weekend", "y_pred_xgb", "recent_realized_vol", "score_risk_adj"]
    feature_cols = [c for c in panel.columns if c not in base_cols + ["fold_id", "split_label", "sign_true", "sign_pred"]]

    wide_parts = []
    for t in ["AMZN", "MSFT"]:
        sub = panel.loc[panel["ticker"] == t, ["date_decision", "y_true_weekend", "y_pred_xgb", "recent_realized_vol", "score_risk_adj"] + feature_cols].copy()
        rename_map = {c: f"{t.lower()}_{c}" for c in sub.columns if c != "date_decision"}
        sub = sub.rename(columns=rename_map)
        wide_parts.append(sub)

    merged = wide_parts[0].merge(wide_parts[1], on="date_decision", how="inner")
    merged = merged.sort_values("date_decision").reset_index(drop=True)

    # Market regime proxy via equal-weight prior-week return sign.
    merged["ew_pred"] = 0.5 * (merged["amzn_y_pred_xgb"] + merged["msft_y_pred_xgb"])
    merged["regime_up"] = (merged["ew_pred"].shift(1) > 0).astype(int)

    return merged

date_panel = to_date_level(xgb_panel)
print(date_panel.shape)
date_panel.head()


In [ ]:
# Train/val/test split aligned with forward chronology.
q_train = date_panel["date_decision"].quantile(0.7)
q_val = date_panel["date_decision"].quantile(0.85)

train_df = date_panel.loc[date_panel["date_decision"] <= q_train].reset_index(drop=True)
val_df = date_panel.loc[(date_panel["date_decision"] > q_train) & (date_panel["date_decision"] <= q_val)].reset_index(drop=True)
test_df = date_panel.loc[date_panel["date_decision"] > q_val].reset_index(drop=True)

print({"train": len(train_df), "val": len(val_df), "test": len(test_df), "train_end": str(q_train.date()), "val_end": str(q_val.date())})


In [ ]:
class WeekendTradingEnv(gym.Env):
    metadata = {"render_modes": ["human"]}

    def __init__(self, df: pd.DataFrame, txn_cost_bps: float = 5.0, dd_penalty: float = 0.0):
        super().__init__()
        self.df = df.reset_index(drop=True)
        self.txn_cost = txn_cost_bps / 10_000.0
        self.dd_penalty = dd_penalty

        self.feature_cols = [
            "amzn_y_pred_xgb", "msft_y_pred_xgb",
            "amzn_recent_realized_vol", "msft_recent_realized_vol",
            "amzn_score_risk_adj", "msft_score_risk_adj",
            "regime_up",
        ]

        self.action_space = spaces.Discrete(4)
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(len(self.feature_cols) + 3,), dtype=np.float32)

        self.reset()

    def _action_to_weights(self, a: int):
        # 0: flat, 1: amzn long, 2: msft long, 3: both long equal
        if a == 0:
            return np.array([0.0, 0.0], dtype=float)
        if a == 1:
            return np.array([1.0, 0.0], dtype=float)
        if a == 2:
            return np.array([0.0, 1.0], dtype=float)
        return np.array([0.5, 0.5], dtype=float)

    def _get_obs(self):
        row = self.df.iloc[self.idx]
        core = row[self.feature_cols].astype(float).values
        extra = np.array([self.prev_pos_amzn, self.prev_pos_msft, self.prev_nav_ret], dtype=float)
        return np.concatenate([core, extra]).astype(np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.idx = 0
        self.nav = 1.0
        self.peak_nav = 1.0
        self.prev_pos_amzn = 0.0
        self.prev_pos_msft = 0.0
        self.prev_nav_ret = 0.0
        self.logs = []
        return self._get_obs(), {}

    def step(self, action):
        row = self.df.iloc[self.idx]

        w = self._action_to_weights(int(action))
        prev_w = np.array([self.prev_pos_amzn, self.prev_pos_msft], dtype=float)

        ret_vec = np.array([row["amzn_y_true_weekend"], row["msft_y_true_weekend"]], dtype=float)
        gross_ret = float(np.dot(w, ret_vec))

        turnover = float(np.abs(w - prev_w).sum())
        cost = self.txn_cost * turnover

        nav_before = self.nav
        net_ret = gross_ret - cost
        self.nav *= (1.0 + net_ret)
        self.peak_nav = max(self.peak_nav, self.nav)
        drawdown = (self.peak_nav - self.nav) / (self.peak_nav + 1e-12)

        reward = net_ret - self.dd_penalty * drawdown

        self.logs.append({
            "date_decision": row["date_decision"],
            "action": int(action),
            "w_amzn": w[0],
            "w_msft": w[1],
            "ret_amzn": ret_vec[0],
            "ret_msft": ret_vec[1],
            "gross_ret": gross_ret,
            "turnover": turnover,
            "cost": cost,
            "net_ret": net_ret,
            "nav_before": nav_before,
            "nav_after": self.nav,
            "drawdown": drawdown,
            "reward": reward,
        })

        self.prev_pos_amzn, self.prev_pos_msft = w[0], w[1]
        self.prev_nav_ret = net_ret

        self.idx += 1
        terminated = self.idx >= len(self.df)
        truncated = False

        obs = self._get_obs() if not terminated else np.zeros(self.observation_space.shape[0], dtype=np.float32)
        return obs, float(reward), terminated, truncated, {}


In [ ]:
def evaluate_policy_in_env(model, env: WeekendTradingEnv):
    obs, _ = env.reset(seed=SEED)
    done = False
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
    return pd.DataFrame(env.logs)

train_env = DummyVecEnv([lambda: WeekendTradingEnv(train_df, txn_cost_bps=5.0, dd_penalty=0.01)])
val_env_raw = WeekendTradingEnv(val_df, txn_cost_bps=5.0, dd_penalty=0.01)
test_env_raw = WeekendTradingEnv(test_df, txn_cost_bps=5.0, dd_penalty=0.01)

ppo = PPO(
    policy="MlpPolicy",
    env=train_env,
    verbose=0,
    seed=SEED,
    learning_rate=3e-4,
    n_steps=128,
    batch_size=64,
    gamma=0.99,
    gae_lambda=0.95,
    ent_coef=0.0,
)

ppo.learn(total_timesteps=30_000)

val_log = evaluate_policy_in_env(ppo, val_env_raw)
test_log = evaluate_policy_in_env(ppo, test_env_raw)


In [ ]:
# Basic RL validation tests.
assert (test_log["gross_ret"] - test_log["cost"] - test_log["net_ret"]).abs().max() < 1e-10, "Reward accounting mismatch"
assert test_log["date_decision"].is_monotonic_increasing, "Temporal order violated"

# Determinism check (same seed, same env, same model deterministic mode).
retest_log = evaluate_policy_in_env(ppo, WeekendTradingEnv(test_df, txn_cost_bps=5.0, dd_penalty=0.01))
assert np.allclose(test_log["action"].values, retest_log["action"].values), "Determinism check failed"

print("RL checks: OK")


In [ ]:
# Persist RL interface contract.
rl_episode_log = test_log.copy()
rl_episode_log.to_parquet(OUT_DIR / "rl_episode_log.parquet", index=False)
val_log.to_parquet(OUT_DIR / "rl_episode_log_val.parquet", index=False)
ppo.save(str(MODEL_DIR / "ppo_weekend_policy"))

print(rl_episode_log.head())
print("saved model + logs")
